# Abha S1 One-Way Candidate Scenarios

This Colab notebook builds **S1A** and **S1B** on top of the existing Abha S0 baseline and displays the **Folium comparison map directly inside Colab**.

In [1]:
!pip install -q osmnx folium geopandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 5.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
%cd "/content/drive/MyDrive/Trafic_AI_Opti"

!ls


/content/drive/MyDrive/Trafic_AI_Opti
abha_network_baseline.ipynb  abha_s1_oneway_colab.ipynb   Untitled0.ipynb
abha_network_baseline.py     abha_s1_oneway_scenarios.py


In [4]:
import math
from pathlib import Path

import folium
from folium import FeatureGroup
from folium.plugins import Fullscreen, MeasureControl
import numpy as np
import pandas as pd

from abha_network_baseline import (
    ABHA_CENTER_LAT,
    ABHA_CENTER_LON,
    download_abha_osm,
    extract_main_roads,
    build_ring_corridor,
    clean_ring_corridor,
    create_baseline,
)

TARGET_ROAD = "King Abdulaziz Road"
OUT_DIR = Path("data/abha_baseline")

S1_DIRECTIONS = {
    "S1A": {
        "name": "King Abdulaziz One-Way — NE",
        "target_bearing": 45.0,
    },
    "S1B": {
        "name": "King Abdulaziz One-Way — SW",
        "target_bearing": 225.0,
    },
}

print("Imports ready.")


Imports ready.


In [5]:
def unwrap_primary(value):
    return value[0] if isinstance(value, tuple) else value


def calculate_segment_bearing(geometry):
    if geometry is None or geometry.is_empty:
        return np.nan

    if geometry.geom_type == "MultiLineString":
        geometry = max(
            geometry.geoms,
            key=lambda geom: geom.length,
        )

    coords = list(geometry.coords)

    if len(coords) < 2:
        return np.nan

    lon1, lat1 = coords[0]
    lon2, lat2 = coords[-1]

    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)
    delta_lon = math.radians(lon2 - lon1)

    x = math.sin(delta_lon) * math.cos(lat2_rad)

    y = (
        math.cos(lat1_rad) * math.sin(lat2_rad)
        - math.sin(lat1_rad)
        * math.cos(lat2_rad)
        * math.cos(delta_lon)
    )

    bearing = math.degrees(
        math.atan2(x, y)
    )

    return (bearing + 360.0) % 360.0


def angular_difference(a, b):
    return abs(
        (a - b + 180.0) % 360.0 - 180.0
    )


def create_one_way_scenario(
    baseline_network,
    main_roads,
    scenario_id,
    scenario_name,
    target_bearing,
):
    network = baseline_network.copy()

    network["scenario"] = scenario_name
    network["scenario_id"] = scenario_id
    network["target_road"] = False
    network["target_direction"] = False

    target_ids = set(
        main_roads.loc[
            main_roads["display_name"] == TARGET_ROAD,
            "road_segment_id",
        ]
    )

    if not target_ids:
        raise RuntimeError(
            "King Abdulaziz Road was not found."
        )

    target_mask = (
        network["road_segment_id"]
        .isin(target_ids)
    )

    network.loc[
        target_mask,
        "target_road",
    ] = True

    network["segment_bearing"] = (
        network.geometry.apply(
            calculate_segment_bearing
        )
    )

    network["bearing_difference"] = (
        network["segment_bearing"].apply(
            lambda x: (
                angular_difference(
                    x,
                    target_bearing,
                )
                if pd.notna(x)
                else np.nan
            )
        )
    )

    preferred = (
        target_mask
        & (
            network["bearing_difference"]
            <= 90.0
        )
    )

    opposite = (
        target_mask
        & ~preferred
    )

    network.loc[
        preferred,
        "target_direction",
    ] = True

    network.loc[
        target_mask,
        "intervention",
    ] = "King Abdulaziz one-way candidate"

    network.loc[
        target_mask,
        "direction_modified",
    ] = True

    network.loc[
        preferred,
        "road_open",
    ] = True

    network.loc[
        opposite,
        "road_open",
    ] = False

    target = network.loc[
        target_mask
    ].copy()

    opened = target[
        target["road_open"]
    ].copy()

    closed = target[
        ~target["road_open"]
    ].copy()

    summary = pd.DataFrame(
        {
            "metric": [
                "Scenario ID",
                "Scenario Name",
                "Target Road",
                "Target Bearing",
                "Target Road Segments",
                "Open Target Segments",
                "Closed Target Segments",
                "Open Target Length (km)",
                "Closed Target Length (km)",
            ],
            "value": [
                scenario_id,
                scenario_name,
                TARGET_ROAD,
                target_bearing,
                len(target),
                len(opened),
                len(closed),
                round(
                    opened["length"].sum()
                    / 1000,
                    2,
                ),
                round(
                    closed["length"].sum()
                    / 1000,
                    2,
                ),
            ],
        }
    )

    return network, summary


In [11]:
def add_scenario_layer(
    map_obj,
    network,
    scenario_id,
    show,
):
    target = (
        network[
            network["target_road"]
        ]
        .to_crs("EPSG:4326")
        .copy()
    )

    group = FeatureGroup(
        name=f"{scenario_id} — one-way candidate",
        show=show,
    )

    opened = target[
        target["road_open"]
    ]

    closed = target[
        ~target["road_open"]
    ]

    if not opened.empty:
        folium.GeoJson(
            opened,
            style_function=lambda _: {
                "color": "#1a9850",
                "weight": 6,
                "opacity": 0.95,
            },
            tooltip=folium.GeoJsonTooltip(
                fields=[
                    "name",
                    "segment_bearing",
                    "road_open",
                ],
                aliases=[
                    "Road",
                    "Bearing",
                    "Open",
                ],
                localize=True,
            ),
        ).add_to(group)

    if not closed.empty:
        folium.GeoJson(
            closed,
            style_function=lambda _: {
                "color": "#d73027",
                "weight": 6,
                "opacity": 0.95,
                "dashArray": "8,6",
            },
            tooltip=folium.GeoJsonTooltip(
                fields=[
                    "name",
                    "segment_bearing",
                    "road_open",
                ],
                aliases=[
                    "Road",
                    "Bearing",
                    "Open",
                ],
                localize=True,
            ),
        ).add_to(group)

    group.add_to(map_obj)

def create_comparison_map(
    streets,
    corridor,
    s1a,
    s1b,
):
    m = folium.Map(
        location=[
            ABHA_CENTER_LAT,
            ABHA_CENTER_LON,
        ],
        zoom_start=14,
        tiles="OpenStreetMap",
        control_scale=True,
    )

    base = FeatureGroup(
        name="S0 — Full Network",
        show=True,
    )

    folium.GeoJson(
        streets.to_crs(
            "EPSG:4326"
        ),
        style_function=lambda _: {
            "color": "#8c8c8c",
            "weight": 1,
            "opacity": 0.20,
        },
    ).add_to(base)

    base.add_to(m)

    study = FeatureGroup(
        name="S0 — Cleaned Study Corridor",
        show=True,
    )

    folium.GeoJson(
        corridor.to_crs(
            "EPSG:4326"
        ),
        style_function=lambda _: {
            "color": "#ff7f00",
            "weight": 3,
            "opacity": 0.60,
        },
    ).add_to(study)

    study.add_to(m)

    add_scenario_layer(
        m,
        s1a,
        "S1A",
        True,
    )

    add_scenario_layer(
        m,
        s1b,
        "S1B",
        False,
    )

    folium.LayerControl(
        collapsed=False
    ).add_to(m)

    Fullscreen().add_to(m)

    MeasureControl(
        position="topleft"
    ).add_to(m)

    return m


In [12]:
nodes, streets, origins, destinations = (
    download_abha_osm()
)

main_roads = extract_main_roads(
    streets
)

ring_result = build_ring_corridor(
    streets,
    main_roads,
)

ring = unwrap_primary(
    ring_result
)

clean_result = clean_ring_corridor(
    ring
)

corridor = unwrap_primary(
    clean_result
)

baseline_result = create_baseline(
    streets,
    corridor,
    origins,
    destinations,
)

baseline_network = unwrap_primary(
    baseline_result
)

print()
print("Baseline ready.")
print(
    "Cleaned corridor segments:",
    len(corridor),
)


ABHA OSM VEHICULAR NETWORK
Center: (18.2264426, 42.5053914)
Radius: 1500 meters
Network type: drive

[1/3] Downloading Abha driving network...
Nodes: 1,729
Directed road segments: 4,586

[2/3] Downloading buildings for provisional origins...
All OSM buildings found: 330
Final provisional origins: 315

[3/3] Downloading amenities and shops...
Destinations: 56

MAIN CORRIDORS
Named road segments: 137
Target main-road segments found: 104
display_name
King Faisal Road       61
King Abdulaziz Road    29
King Khalid Road       14
Name: count, dtype: int64

RING CORRIDOR CANDIDATE
Candidate segments: 114

CLEANED STUDY CORRIDOR
Final cleaned segments: 44

S0 BASELINE DATA READINESS
name      : 27/44 (61.4%)
lanes     : 0/44 (0.0%)
maxspeed  : 0/44 (0.0%)
oneway    : 44/44 (100.0%)

Baseline ready.
Cleaned corridor segments: 44


In [13]:
s1a, s1a_summary = (
    create_one_way_scenario(
        baseline_network,
        main_roads,
        "S1A",
        S1_DIRECTIONS["S1A"]["name"],
        S1_DIRECTIONS["S1A"]["target_bearing"],
    )
)

s1b, s1b_summary = (
    create_one_way_scenario(
        baseline_network,
        main_roads,
        "S1B",
        S1_DIRECTIONS["S1B"]["name"],
        S1_DIRECTIONS["S1B"]["target_bearing"],
    )
)

print("S1A SUMMARY")
display(s1a_summary)

print()
print("S1B SUMMARY")
display(s1b_summary)


S1A SUMMARY


,metric,value
0,Scenario ID,S1A
1,Scenario Name,King Abdulaziz One-Way — NE
2,Target Road,King Abdulaziz Road
3,Target Bearing,45.0
4,Target Road Segments,29
5,Open Target Segments,13
6,Closed Target Segments,16
7,Open Target Length (km),3.02
8,Closed Target Length (km),3.17



S1B SUMMARY


,metric,value
0,Scenario ID,S1B
1,Scenario Name,King Abdulaziz One-Way — SW
2,Target Road,King Abdulaziz Road
3,Target Bearing,225.0
4,Target Road Segments,29
5,Open Target Segments,16
6,Closed Target Segments,13
7,Open Target Length (km),3.17
8,Closed Target Length (km),3.02


## Interactive Folium Map

Use the layer control on the top-right to switch between S1A and S1B.

In [14]:
comparison_map = create_comparison_map(
    streets,
    corridor,
    s1a,
    s1b,
)

comparison_map


Output hidden; open in https://colab.research.google.com to view.

In [15]:
OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

s1a_summary.to_csv(
    OUT_DIR / "s1a_oneway_summary.csv",
    index=False,
)

s1b_summary.to_csv(
    OUT_DIR / "s1b_oneway_summary.csv",
    index=False,
)

s1a[
    s1a["target_road"]
].to_crs(
    "EPSG:4326"
).to_file(
    OUT_DIR / "s1a_king_abdulaziz.geojson",
    driver="GeoJSON",
)

s1b[
    s1b["target_road"]
].to_crs(
    "EPSG:4326"
).to_file(
    OUT_DIR / "s1b_king_abdulaziz.geojson",
    driver="GeoJSON",
)

comparison_map.save(
    OUT_DIR
    / "abha_s1_oneway_comparison_map.html"
)

print("Saved outputs to:")
print(OUT_DIR)

print()
print("Generated:")
print("- s1a_oneway_summary.csv")
print("- s1b_oneway_summary.csv")
print("- s1a_king_abdulaziz.geojson")
print("- s1b_king_abdulaziz.geojson")
print("- abha_s1_oneway_comparison_map.html")


Saved outputs to:
data/abha_baseline

Generated:
- s1a_oneway_summary.csv
- s1b_oneway_summary.csv
- s1a_king_abdulaziz.geojson
- s1b_king_abdulaziz.geojson
- abha_s1_oneway_comparison_map.html
